# Práctica 1 · Univariada

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lautaromgo/estadistica-aplicada-unne/blob/main/notebooks/practica/practica_1_univariada.ipynb)

**Cómo funciona esta notebook.** Hay cinco funciones vacías con un `TODO` adentro. Arriba de
cada una está qué mide y para qué sirve; abajo, el paso a paso. Vos escribís el cuerpo, y la
celda que sigue compara tu resultado con el de `pandas` y te dice si está bien.

Las pistas son el paso a paso, no el código. Si una función te sale en una línea copiada de
internet, funciona igual pero no sirve para nada: la idea es escribirlas una vez a mano para
dejar de usarlas como cajas negras.

No hace falta instalar nada ni bajar ningún archivo: la celda de setup lo resuelve sola, acá
y en Colab. Corré todo de arriba hacia abajo.

## 0 · Setup

Corré esta celda y seguí. Carga el dataset, define los colores del curso y la función
`verificar()`, que es la que te va a ir diciendo si cada ejercicio está bien.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Los colores del curso, para que esto se vea igual que las slides.
AZUL, ROJO, AZUL_PALE = "#1E579B", "#D1495B", "#DCEAF7"

# El dataset: precios de vivienda en California, censo de 1990.
# Si tenés el archivo al lado lo usa; si no, lo baja. Así corre igual en tu
# máquina y en Colab, sin cambiar nada.
URL = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
       "master/datasets/housing/housing.csv")
RUTAS = ["../../data/raw/housing.csv", "../data/raw/housing.csv", "housing.csv"]

ruta = next((r for r in RUTAS if os.path.exists(r)), None)
df = pd.read_csv(ruta) if ruta else pd.read_csv(URL)

# La derivada que vamos a usar para forzar los conceptos.
df["ambientes_por_hogar"] = df["total_rooms"] / df["households"]


def verificar(nombre, obtenido, esperado, tol=1e-6):
    """Compara tu resultado con el de pandas. Avisa y sigue: no corta la notebook."""
    try:
        ok = bool(np.isclose(float(obtenido), float(esperado), rtol=tol, atol=tol))
    except (TypeError, ValueError):
        ok = False
    print(f"✅ {nombre}" if ok
          else f"❌ {nombre} → obtuviste {obtenido}, se esperaba {esperado}")


def nro(x, decimales=0):
    """Formatea un número como lo escribimos nosotros: 20.640 · 0,98"""
    return f"{x:,.{decimales}f}".replace(",", "|").replace(".", ",").replace("|", ".")


print(f"{nro(len(df))} filas × {df.shape[1]} columnas   |   origen: {ruta or 'la web'}")

## 1 · El centro: ¿alrededor de qué número viven los datos?

La variable de esta práctica es **`median_income`**: el ingreso mediano de los hogares del
barrio. Viene en **decenas de miles de dólares de 1990**, así que un 3,5 son 35.000 dólares
al año. Es una unidad rara y conviene tenerla presente: los números que vas a ver son chicos,
pero no son porcentajes.

Hay dos formas de contestar *"¿cuál es el valor típico?"*:

- **la media** — sumar todo y repartir en partes iguales. Usa *todos* los valores, y por eso
  cualquier valor extremo tira de ella.
- **la mediana** — ordenar y pararse en el medio. Sólo le importa el orden, así que un valor
  extremo la corre un lugar, no la arrastra.

Las dos son "el centro". Cuál reportás depende de la forma de la distribución, y eso se
decide mirando, no por costumbre.

In [ ]:
ingreso = df["median_income"]

print(ingreso.describe().round(3).to_string())
print(f"\nLa mediana, traducida a dólares al año: {nro(ingreso.median() * 10_000)}")

**Ejercicio 1 y 2.** Escribí `media()` y `mediana()`. Las dos reciben un array o una Series y
devuelven un número.

In [ ]:
def media(valores):
    """El promedio: la suma de todos los valores dividida por cuántos son.

    Es el punto de equilibrio de la distribución. Si los datos fueran pesos sobre una
    regla, la media es donde hay que poner el dedo para que no se caiga — y por eso un
    solo valor muy lejano la corre.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá los valores a array de float y sacá los NaN:
    #        x = np.asarray(valores, dtype=float)  y después  x = x[~np.isnan(x)]
    #   2. Si no quedó ningún valor, devolvé np.nan.
    #   3. Devolvé la suma dividida por la cantidad. Sin .mean() ni np.mean().
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar media(). Mirá el TODO justo arriba.")

In [ ]:
def mediana(valores):
    """El valor del medio, una vez que los datos están ordenados.

    Con n impar hay un valor justo en el medio. Con n par hay dos, y la mediana es el
    promedio de esos dos. Nada más que eso: por eso no le importa cuán grande es el
    valor más grande, sólo que esté arriba.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá a array de float, sacá los NaN y ordenalo con np.sort().
    #   2. Si no quedó nada, devolvé np.nan.
    #   3. Si n es impar (n % 2 == 1), devolvé el valor de la posición n // 2.
    #   4. Si n es par, promediá los de las posiciones n // 2 - 1 y n // 2.
    #      Ojo: Python cuenta desde 0. Con n = 4, el medio son las posiciones 1 y 2.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar mediana(). Mirá el TODO justo arriba.")

In [ ]:
verificar("media",   media(ingreso),   ingreso.mean())
verificar("mediana", mediana(ingreso), ingreso.median())

# Y sobre la única columna con nulos, para ver que los estás descartando bien:
verificar("media con nulos", media(df["total_bedrooms"]), df["total_bedrooms"].mean())

#### ⚠️ Probalo y mirá (1): quién aguanta los extremos

`ambientes_por_hogar` (`total_rooms / households`) llega a **141,9** ambientes por hogar,
contra una mediana de 5,2. Son **27 veces** la mediana, y no es una mansión: es un barrio
entero mal dividido.

Sacale el **0,1% más alto** —veintiuna filas de 20.640— y mirá cuánto se mueve cada medida.

In [ ]:
amb = df["ambientes_por_hogar"]
corte = amb.quantile(0.999)              # el 0,1% más alto
recortado = amb[amb <= corte]

tabla = pd.DataFrame({
    "con todo":     [media(amb),       mediana(amb),       amb.std()],
    "sin el 0,1%":  [media(recortado), mediana(recortado), recortado.std()],
}, index=["media", "mediana", "desvío"])
tabla["cambio %"] = (tabla["sin el 0,1%"] / tabla["con todo"] - 1) * 100

print(f"filas recortadas: {(amb > corte).sum()} de {nro(len(amb))}\n")
tabla.round(3)

**El centro aguanta; la dispersión no.** Veintiuna filas de veinte mil y pico mueven la media
un **0,91%** y la mediana un **0,04%** —o sea, nada—, pero se llevan puesto el **31%** del
desvío.

Y tiene sentido: la media reparte el exceso entre 20.640 casos y se diluye. El desvío, en
cambio, mide *distancias al cuadrado*, así que un valor a 136 unidades del centro entra al
cálculo elevado al cuadrado y pesa él solo más que miles de valores normales.

> **La regla:** que la media aguante no quiere decir que el resumen esté bien. Con n grande el
> centro casi no se mueve y la variable sigue teniendo barrios imposibles adentro. Por eso el
> centro nunca se reporta solo.

## 2 · La dispersión: ¿qué tan lejos del centro vive cada uno?

Dos barrios pueden tener el mismo ingreso medio y ser completamente distintos: uno donde todos
ganan parecido, y otro donde la mitad gana muy poco y la otra mitad mucho. El centro no los
distingue. La dispersión sí.

La idea es medir **cuánto se aleja cada valor de la media**. El problema es que esas
distancias, sumadas tal cual, dan siempre cero: las de arriba cancelan las de abajo. Por eso se
elevan al cuadrado antes de promediar, y eso es la **varianza**.

Como la varianza queda en unidades al cuadrado —¿"decenas de miles de dólares al cuadrado"?—,
se le toma la raíz para volver a la unidad original. Eso es el **desvío estándar**, y es el
número que se reporta.

In [ ]:
def varianza(valores, muestra=True):
    """El promedio de las distancias al cuadrado respecto de la media.

        muestra=True   divide por (n-1)   [varianza muestral, la que usa pandas]
        muestra=False  divide por n       [varianza poblacional, la que usa numpy]

    Cuál corresponde depende de si tus datos SON la población o son una muestra de algo
    más grande. Cuánto cambia el resultado se mide dos celdas más abajo.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá a array de float y sacá los NaN.
    #   2. Si quedan menos de 2 valores no hay dispersión que medir: devolvé np.nan.
    #   3. Calculá la media. Podés usar tu propia media(): para eso la escribiste.
    #   4. Restale la media a cada valor, elevá al cuadrado y sumá. Sin .var().
    #   5. Dividí por (n - 1) si muestra=True, o por n si muestra=False.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar varianza(). Mirá el TODO justo arriba.")

In [ ]:
def desvio_estandar(valores, muestra=True):
    """La raíz cuadrada de la varianza.

    Existe sólo para eso: devolver el número a la unidad de los datos. Un desvío de 1,9
    sobre median_income son 19.000 dólares al año, algo que podés decir en voz alta. Una
    varianza de 3,61 está en "decenas de miles de dólares al cuadrado" y no significa nada.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Una línea: np.sqrt() de tu varianza().
    #   2. Acordate de pasarle el parámetro `muestra`, o va a ignorar lo que le pidan.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar desvio_estandar(). Mirá el TODO justo arriba.")

In [ ]:
# Ojo con un detalle que confunde a todo el mundo: pandas usa n-1 por defecto
# en .var() y .std(); numpy usa n en np.var(). No es un error de nadie, son
# defaults distintos. Si comparás resultados con un compañero, empezá por ahí.
verificar("varianza muestral",    varianza(ingreso),                ingreso.var())
verificar("varianza poblacional", varianza(ingreso, muestra=False), np.var(ingreso))
verificar("desvío",               desvio_estandar(ingreso),         ingreso.std())

**¿Y cuándo importa el `n−1`?** Casi nunca… hasta que importa muchísimo.

In [ ]:
isla = df.loc[df["ocean_proximity"] == "ISLAND", "median_income"]

for nombre, v in [("todo el dataset", ingreso), ("ISLAND", isla)]:
    muestral = varianza(v)
    poblacional = varianza(v, muestra=False)
    print(f"{nombre:16s} n = {nro(len(v)):>6}  |  muestral {muestral:7.4f}   "
          f"poblacional {poblacional:7.4f}   |  diferencia {(muestral / poblacional - 1) * 100:5.2f}%")

Con 20.640 filas, dividir por `n` o por `n−1` cambia el resultado un **0,00%**: es
literalmente indistinguible. Con los **cinco** barrios de `ISLAND`, la diferencia es del
**25%**.

> **La regla:** el `n−1` no es una sutileza de examen. Es una corrección que sólo se nota
> cuando tenés pocos datos, que es justo cuando todo lo demás también se vuelve frágil.

## 3 · Comparar dispersiones: el coeficiente de variación

Un desvío de 1,9 ¿es mucho? No se puede contestar sin saber sobre qué. Sobre una media de 3,9
es enorme; sobre una media de 400 es nada.

El **coeficiente de variación** (CV) resuelve eso dividiendo el desvío por la media: queda un
número **sin unidades**, y recién ahí se puede comparar la variabilidad de cosas que están en
escalas distintas.

In [ ]:
def coef_variacion(valores):
    """El desvío dividido por la media: dispersión RELATIVA al tamaño de lo que se mide.

    Al no tener unidades, permite comparar la variabilidad de un ingreso con la de un
    precio, o la de un grupo grande con la de uno chico.

    DÓNDE SE ROMPE: divide por la media. Si la media es cero no está definido, y si la
    variable tiene valores negativos la media puede quedar cerca de cero y el CV se
    dispara sin significar nada. Sólo tiene sentido en variables con cero absoluto:
    importes, cantidades, precios.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Pasá a array, sacá los NaN y calculá la media con tu función.
    #   2. Si la media es 0, devolvé np.nan: no se puede dividir, y no hay CV que valga.
    #   3. Devolvé desvio_estandar(x) dividido por esa media.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar coef_variacion(). Mirá el TODO justo arriba.")

In [ ]:
verificar("cv del ingreso", coef_variacion(ingreso),
          ingreso.std() / ingreso.mean())
verificar("cv del precio",  coef_variacion(df["median_house_value"]),
          df["median_house_value"].std() / df["median_house_value"].mean())

#### ⚠️ Probalo y mirá (2): el segmento más "parejo" tiene cinco casos

Calculá el CV del ingreso **dentro de cada segmento** y ordenalos de menor a mayor. Antes de
correr la celda, apostá cuál va a salir primero.

In [ ]:
filas = []
for grupo, sub in df.groupby("ocean_proximity"):
    filas.append({
        "segmento": grupo,
        "n": len(sub),
        "media": media(sub["median_income"]),
        "desvío": desvio_estandar(sub["median_income"]),
        "cv": coef_variacion(sub["median_income"]),
    })

tabla_cv = pd.DataFrame(filas).set_index("segmento").sort_values("cv")
tabla_cv.round(4)

`ISLAND` gana cómodo: **CV 0,162**, contra 0,45–0,50 de todos los demás. Leído sin mirar el
`n`, eso es un hallazgo: *las islas son el segmento con los ingresos más parejos de
California*.

Ahora mirá la columna `n`. Son **cinco barrios**. Cinco valores parecidos entre sí no son un
segmento homogéneo: son cinco valores. Que salgan parecidos es perfectamente posible por azar,
y el CV no tiene forma de avisarte — es una división, hace la cuenta que le pidas.

> **La regla:** ninguna medida de dispersión se lee sin el `n` al lado. El CV compara escalas
> distintas; no compara cuánto podés confiar.

## 4 · Todo junto, en un gráfico

No hay nada nuevo acá: es para ver de una lo que vinieron diciendo los números. Esta celda ya
está resuelta, sólo corrila.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.hist(ingreso, bins=60, color=AZUL_PALE, edgecolor=AZUL, linewidth=0.5)
ax.axvline(media(ingreso),   color=ROJO,      ls="--", lw=2,
           label=f"media = {media(ingreso):.2f}")
ax.axvline(mediana(ingreso), color="seagreen", ls="--", lw=2,
           label=f"mediana = {mediana(ingreso):.2f}")

ax.set_title("median_income — ingreso mediano del barrio, en decenas de miles de USD")
ax.set_xlabel("median_income")
ax.set_ylabel("barrios")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

La media queda a la derecha de la mediana: la cola larga hacia los ingresos altos tira de
ella. Es la misma asimetría que ya viste en el precio.

Y ese pico solitario pegado al borde derecho no es una cola: son **49 barrios con exactamente
15,0001**. El mismo tope que le vimos al precio en el punto 1, ahora en el ingreso.

## ✍️ Para pensar

Dos preguntas. Se contestan en dos o tres líneas cada una, con lo que viste acá.

1. Te piden **el ingreso típico de un barrio de California** en una sola línea, para un
   informe. ¿Reportás la media (3,87) o la mediana (3,53)? Justificá con algo que hayas visto
   en esta notebook.

2. Un compañero te dice: *"`ISLAND` es el segmento más homogéneo de California, su CV es menos
   de la mitad que el del resto"*. ¿Qué le contestás?

*Escribí tus respuestas acá (doble clic para editar esta celda).*

**1.**

**2.**